# 18 | Out-of-Fold CNN Confidence and Error Audit

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Research question:** How confident is the existing 1,000-cap Weighted CNN when correct or incorrect, especially on Shuffling and stair activities? Use exactly 400 optimiser steps across three participant-independent GroupKFold folds, preserving Stage 17's architecture and training policy. These are **uncalibrated softmax** diagnostics on development data, not independent testing or a clinical uncertainty estimate.

## 01. Imports and reusable code

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score

# Register reusable core and HARTH study-specific functions.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root:
        raise RuntimeError("Open this notebook inside the repository.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, prepare_harth_nested_window_cohort
from core.temporal_cnn import (
    train_temporal_cnn_fixed_steps, predict_temporal_cnn_probabilities,
)
from core.prediction_reliability import audit_prediction_confidence

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 02. Reconstruct the Stage 17 development dataset

Use the identical nested recording adapter and source class definitions. Only the base-500 rows may enter Validation; 1,000-cap samples are available only to Train participants. The original Test IDs remain excluded.

In [ ]:
# Preserve the same protected subjects and documented eight-class task.
RESERVED_TEST = {"S023", "S032"}
SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
X, metadata, overview = prepare_harth_nested_window_cohort(
    subjects=SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    base_cap=500, expanded_cap=1000,
    window_size=100, step_size=50, sampling_rate=50,
)
base_rows = np.flatnonzero(metadata["in_base_500"].to_numpy(dtype=bool))
y = metadata["label"].to_numpy()
group_ids = metadata["subject_id"].to_numpy()
assert len(SUBJECTS) == 29 and len(base_rows) == 14201
assert not (set(group_ids) & RESERVED_TEST)
print("Expanded input:", X.shape, "| Original Validation candidates:", len(base_rows))

## 03. Train the predeclared CNN in each development fold

Each fold has a new model, the same **400-step fixed budget** and inverse-square-root class weighting based only on Fold Train. Validation softmax predictions are collected at the final step, with the original source-row ID.

In [ ]:
# Produce aligned out-of-fold probabilities without touching reserved Test.
FOLDS, STEPS = 3, 400
device = "cuda" if torch.cuda.is_available() else "cpu"
folds = GroupKFold(n_splits=FOLDS)
records, fold_metrics = [], []
for fold, (tr, va) in enumerate(
    folds.split(np.zeros(len(base_rows)), y[base_rows], groups=group_ids[base_rows]),
    start=1,
):
    base_train = base_rows[tr]
    validation = base_rows[va]
    train_subjects = set(group_ids[base_train])
    validation_subjects = set(group_ids[validation])
    assert not (train_subjects & validation_subjects)
    train_rows = np.flatnonzero(np.isin(group_ids, list(train_subjects)))

    # Fit 1,000-cap Weighted CNN on Train people only.
    model, history, predicted, train_stats = train_temporal_cnn_fixed_steps(
        X_train=X[train_rows], y_train=y[train_rows],
        X_validation=X[validation], y_validation=y[validation],
        classes=TASK_CLASSES, max_steps=STEPS, eval_every=100,
        batch_size=128, learning_rate=1e-3, seed=42,
        device=device, class_weighting="inverse_sqrt",
    )
    probabilities = predict_temporal_cnn_probabilities(
        model, X[validation], train_stats, TASK_CLASSES,
        batch_size=256, device=device,
    )
    assert np.array_equal(
        np.asarray(TASK_CLASSES)[probabilities.argmax(axis=1)], predicted
    ), "Probability inference must match the final-step prediction."
    records.append(pd.DataFrame({
        "fold": fold, "source_row": validation,
        "subject_id": group_ids[validation],
        "true_label": y[validation],
        **{f"prob_{code}": probabilities[:, i]
           for i, code in enumerate(TASK_CLASSES)},
    }))
    fold_metrics.append({
        "fold": fold, "validation_subjects": len(validation_subjects),
        "validation_windows": len(validation),
        "macro_f1": f1_score(
            y[validation], predicted, labels=TASK_CLASSES,
            average="macro", zero_division=0,
        ),
    })
    print("Completed fold", fold)
oof = pd.concat(records, ignore_index=True).sort_values("source_row")
display(pd.DataFrame(fold_metrics).round(4))

## 04. Verify out-of-fold provenance and compute descriptive confidence metrics

Each base-500 development row must appear in precisely one held-out fold. The probability table has columns in the fixed original activity-code order.

In [ ]:
# Audit probability rows before drawing any inference.
assert len(oof) == len(base_rows)
assert oof["source_row"].is_unique
np.testing.assert_array_equal(oof["source_row"].to_numpy(), np.sort(base_rows))
np.testing.assert_array_equal(
    oof["true_label"].to_numpy(), y[np.sort(base_rows)]
)
prob_columns = [f"prob_{code}" for code in TASK_CLASSES]
details, reliability, by_subject, summary = audit_prediction_confidence(
    y_true=oof["true_label"].to_numpy(),
    probabilities=oof[prob_columns].to_numpy(),
    subjects=oof["subject_id"].to_numpy(),
    classes=TASK_CLASSES, n_bins=10,
)
details["fold"] = oof["fold"].to_numpy()
details["source_row"] = oof["source_row"].to_numpy()
display(pd.DataFrame([summary]).round(4))
display(reliability.round(4))
print("Note: uncalibrated softmax, not validated uncertainty.")

## 05. Reliability diagram and confidence distribution

Compare average softmax confidence with empirical accuracy in fixed-width bins. These observations are correlated within recordings; ECE is descriptive and sensitive to binning.

In [ ]:
# Plot observed accuracy versus softmax confidence for non-empty bins.
used = reliability.loc[reliability["windows"] > 0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration reference")
axes[0].plot(
    used["mean_confidence"], used["observed_accuracy"],
    marker="o", label="Observed development windows",
)
axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean softmax confidence",
            ylabel="Observed accuracy", title="Out-of-fold reliability diagram")
axes[0].legend()
axes[0].grid(alpha=0.2)

# Contrast confidence of correct and incorrect windows.
correct = details.loc[details["correct"], "confidence"]
wrong = details.loc[~details["correct"], "confidence"]
axes[1].hist([correct, wrong], bins=np.linspace(0, 1, 21),
             label=["Correct", "Incorrect"], alpha=0.7)
axes[1].set(xlabel="Maximum softmax probability", ylabel="Windows",
            title="Confidence on correct versus incorrect predictions")
axes[1].legend()
fig.tight_layout()
plt.show()

## 06. Audit Shuffling, stairs and confident misclassifications

Prior stages found uneven class recall. Report support, accuracy and wrong predictions with high softmax values separately, without treating confidence as a clinically actionable probability.

In [ ]:
# Inspect held-out class-specific errors, keeping original code meanings.
class_table = []
for code in TASK_CLASSES:
    part = details.loc[details["true_label"] == code]
    mistakes = part.loc[~part["correct"]]
    class_table.append({
        "class": ACTIVITY_NAMES[code], "support": len(part),
        "recall": float(part["correct"].mean()) if len(part) else np.nan,
        "errors": len(mistakes),
        "high_confidence_errors_0_90":
            int((mistakes["confidence"] >= 0.90).sum()),
        "mean_error_confidence":
            float(mistakes["confidence"].mean()) if len(mistakes) else np.nan,
    })
class_report = pd.DataFrame(class_table)
display(class_report.round(4))
display(by_subject.sort_values("high_confidence_errors", ascending=False).round(4))

# Display actual source-label confusion pairs among high-confidence errors.
confident_errors = details.loc[
    (~details["correct"]) & (details["confidence"] >= 0.9)
]
confusion_pairs = (confident_errors
    .groupby(["true_label", "predicted_label"])
    .size().rename("windows").reset_index()
    .sort_values("windows", ascending=False))
display(confusion_pairs.head(15))

## 07. Quality gate and interpretation

The current study evaluates **uncalibrated** probability scores, not the efficacy of confidence-based refusal or an uncertainty-aware decision policy. No temperature scaling, threshold tuning or external calibration is performed.

In [ ]:
# Confirm exact fold coverage and an untouched protected Test boundary.
assert len(details) == 14201
assert oof["source_row"].is_unique
assert details["subject_id"].nunique() == 29
assert not (set(details["subject_id"]) & RESERVED_TEST)
assert int(reliability["windows"].sum()) == len(details)
assert np.allclose(oof[prob_columns].sum(axis=1), 1, atol=1e-5)
assert 0 <= summary["accuracy"] <= 1
assert 0 <= summary["ece_10_bins"] <= 1
print("Out-of-fold softmax confidence audit: PASSED")
print("Original reserved Test used: NO")
print("Calibration or clinical uncertainty validity established: NO")